# Device characterization

Original characterization workflow: activation profiles, ON/OFF episodes, time and power weights, and minute/block comparisons. Run from `Code/` using the datasets in `../Data/`. Figures and tables are displayed without exporting files.


In [ ]:
import openpyxl
import pyomo.environ as pe
import numpy as np
import os
import pandas as pd
from datetime import datetime
from datetime import timedelta
import matplotlib.pyplot as plt
from dataclasses import dataclass
import itertools
from matplotlib.patches import Patch
from pathlib import Path




# Consumption Data

In [ ]:
Year=2019
Month=2
Day=14
N_DEV = 13

# instante inicial e final do dia em análise
day_start_dt = datetime(Year, Month, Day, 0, 0, 0)
day_end_dt = day_start_dt + timedelta(days=1)

# =========================
# DEVICE GROUPS
# =========================
# Non-Operational: caracterização por episódios ao longo do dia inteiro.
NON_OPERATIONAL_DEVICES = {0, 1, 2, 3, 4}  # Drinks Refrigerator, Refrigerators 2–4, Garde Manger
# Operational: episódios apenas entre a primeira e a última amostra ON.
OPERATIONAL_DEVICES = {5, 6}  # Dual Fryer, Salamander
# Duração em minutos usada quando falta um estado em toda a janela.
MISSING_STATE_DURATION = 1e-6
NON_FLEXIBLE_DEVICES = {7, 8, 9}  # Glass Washer, Dish Washer, Convection Oven
POWER_FLEXIBLE_DEVICES = {10, 11, 12}  # Light1, Light2, Exhaust
EXTERNAL_PROFILE_DEVICES = {12}  # Exhaust

POWER_FLEXIBLE_SCHEDULES = {
    10: {
        "name": "Light1",
        "start": 10 * 60,          # 10:00
        "end": 22 * 60 + 30        # 22:30
    },
    11: {
        "name": "Light2",
        "start": 10 * 60,          # 10:00
        "end": 22 * 60 + 30        # 22:30
    }
}
EXTERNAL_PROFILE_DATES = {
    12: "2019-03-11"
}

N_MIN_ACTIVATIONS = 2
T_NO_ACTIVITY_FALLBACK = 5000
T_MIN = 5
T_OFF_FORCED_ON = 5
DAY_MINUTES = 24 * 60
T= DAY_MINUTES

# diretório dos ficheiros
diretorio = '../Data'

# mapeamento appliance -> ficheiro de ativações
activation_files = {
    0: "activations_Drinks_Refrigerator.csv",
    1: "activations_Refrigerator_2.csv",
    2: "activations_Refrigerator_3.csv",
    3: "activations_Refrigerator_4.csv",
    4: "activations_Garde_Manger_10.csv",
    5: "activations_Dual_Fryer.csv",
    6: "activations_Salamander_1.csv",
    7: "activations_Glass_Washer.csv",
    8: "activations_Dish_Washer.csv",
    9: "activations_Convection_Oven_1.csv",
}
external_activation_files = {
    12: "05_Exhaust_activations.xlsx"
}

# Índices de base zero, na mesma ordem das linhas do Info.xlsx.
DEVICE_NAMES = {
    0: "Drinks Refrigerator",
    1: "Refrigerator 2",
    2: "Refrigerator 3",
    3: "Refrigerator 4",
    4: "Garde Manger",
    5: "Dual Fryer",
    6: "Salamander",
    7: "Glass Washer",
    8: "Dish Washer",
    9: "Convection Oven",
    10: "Light1",
    11: "Light2",
    12: "Exhaust"
}

TIMEZONE_LIMITS = {
    0: (0, 600),
    1: (600, 1170),
    2: (1170, 1350),
    3: (1350, 1440),
}
TIMEZONE_LABELS = {
    0: "Residual Consumption",
    1: "Preparation",
    2: "Dinner Service",
    3: "Closing"
}


In [ ]:
# ======================================
# LEITURA DO INFO.XLSX
# ======================================

df_info = pd.read_excel('../Data/Info.xlsx')

print("Colunas do Info.xlsx:")
print(df_info.columns.tolist())

# se quiseres confirmar o número de equipamentos do Excel
print("\nNúmero de linhas no Info.xlsx:", len(df_info))

# opcional: garantir consistência
if len(df_info) != N_DEV:
    raise ValueError(f"N_DEV={N_DEV}, mas o Info.xlsx tem {len(df_info)} linhas.")

# parâmetros por equipamento
dev_names = df_info["Dev"].tolist()
if dev_names != [DEVICE_NAMES[j] for j in range(N_DEV)]:
    raise ValueError("A ordem dos equipamentos no Info.xlsx não coincide com DEVICE_NAMES.")

# índice do tipo de flexibilidade
Type = df_info["Type"].to_numpy(dtype=int)

# estado inicial
# para já vamos ler do Excel; mais tarde podes forçar a zero ou atualizar dinamicamente
State_Init = df_info["State_Init"].to_numpy(dtype=int)

# potência máxima
Power_max = df_info["Power_max"].to_numpy(dtype=float)

# pesos da componente de potência
Weight_P_max = df_info["Weight_P_max"].to_numpy(dtype=float)
Weight_P_min = df_info["Weight_P_min"].to_numpy(dtype=float)

# percentagens de potência
# MUITO IMPORTANTE: converter de % para fração
Y_max = df_info["Y_max (%)"].to_numpy(dtype=float) / 100.0
Y_min = df_info["Y_min (%)"].to_numpy(dtype=float) / 100.0

# tempo mínimo para manter alteração de potência
TimeHoldPower = df_info["TimeHoldPower"].to_numpy(dtype=float)

# impressão de controlo
print("\nResumo dos equipamentos:")
for j in range(N_DEV):
    print(
        f"{j}: {dev_names[j]} | "
        f"Type={Type[j]} | "
        f"State_Init={State_Init[j]} | "
        f"Pmax={Power_max[j]} | "
        f"Ymin={Y_min[j]} | "
        f"Ymax={Y_max[j]}"
        f" | TimeHoldPower={TimeHoldPower[j]}"
    )

In [ ]:
def minute_of_day(ts):
    m = ts.hour * 60 + ts.minute
    if ts.second >= 30:
        m += 1
    return min(m, 1439)

def get_time_zone(minute):
    """
    0 -> 00:00 - 09:59
    1 -> 10:00 - 19:29
    2 -> 19:30 - 22:29
    3 -> 22:30 - 23:59
    """
    if 0 <= minute <= 599:
        return 0
    elif 600 <= minute <= 1169:
        return 1
    elif 1170 <= minute <= 1349:
        return 2
    else:
        return 3

def load_activation_day_csv(diretorio, nome_do_arquivo, target_date):
    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)
    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    # Os CSVs originais usam ';'; Refrigerators 2–4 usam ',' e maiúsculas.
    df = pd.read_csv(caminho_arquivo, sep=None, engine="python").iloc[:, 0:3].copy()
    df.columns = df.columns.str.strip().str.lower()

    df["first_min_on"] = pd.to_datetime(df["first_min_on"], utc=True, errors="coerce")
    df["last_min_on"]  = pd.to_datetime(df["last_min_on"],  utc=True, errors="coerce")

    df["first_min_on"] = df["first_min_on"].dt.tz_convert(None)
    df["last_min_on"]  = df["last_min_on"].dt.tz_convert(None)

    df = df.dropna(subset=["first_min_on", "last_min_on"]).sort_values("first_min_on").reset_index(drop=True)

    target_date = pd.to_datetime(target_date).date()
    df_day = df[df["first_min_on"].dt.date == target_date].copy().reset_index(drop=True)

    return df_day
def build_X_day_for_device(j, nome_do_arquivo, diretorio, target_date, X_day):
    """
    Constrói X_day[:, j] para o dia inteiro.
    """
    X_day[:, j] = 0

    df_day = load_activation_day_csv(diretorio, nome_do_arquivo, target_date)

    if df_day.empty:
        return

    for _, row in df_day.iterrows():
        start = minute_of_day(row["first_min_on"])
        end   = minute_of_day(row["last_min_on"])

        if end >= start:
            X_day[start:end + 1, j] = 1
        else:
            X_day[start:1440, j] = 1
            X_day[0:end + 1, j] = 1

In [ ]:
# 1) Construir X_day global apenas para devices com ficheiro
X_day = np.zeros((T, N_DEV), dtype=int)

for j, nome_do_arquivo in activation_files.items():
    build_X_day_for_device(
        j=j,
        nome_do_arquivo=nome_do_arquivo,
        diretorio=diretorio,
        target_date=day_start_dt.date(),
        X_day=X_day
    )

# 2) Inicializar matrizes
X_onoff_FC_matrix = np.zeros((T, N_DEV), dtype=int)

target_date = day_start_dt.date()

# 3) Preencher baseline com ativações reais
X_onoff_FC_matrix[:, :] = X_day[:T, :]

print("\nX_onoff_FC_matrix (baseline) for the day:")
print(X_onoff_FC_matrix)

# Auxiliary functions

In [ ]:
def plot_power_flexible_device_weight_and_activation(
    device,
    W_matrix,
    X_matrix,
    Y_matrix,
    dev_names,
    characterization_mode,
    Type,
    W_P_min,
    W_P_max,
    figsize=(16, 5)
):
    """
    Plota um equipamento power-flexible:
        - W_P no eixo esquerdo;
        - W_P,min e W_P,max tracejados;
        - ON/OFF state como barras azuis;
        - Y como linha laranja;
        - eixo temporal em horas.
    """

    W_matrix = np.asarray(W_matrix, dtype=float)
    X_matrix = np.asarray(X_matrix)
    Y_matrix = np.asarray(Y_matrix, dtype=float)
    Type = np.asarray(Type)

    if W_matrix.ndim != 2:
        raise ValueError("W_matrix deve ter shape (T, N_DEV).")

    if X_matrix.shape != W_matrix.shape:
        raise ValueError("X_matrix deve ter o mesmo shape de W_matrix.")

    if Y_matrix.shape != W_matrix.shape:
        raise ValueError("Y_matrix deve ter o mesmo shape de W_matrix.")

    if len(dev_names) != W_matrix.shape[1]:
        raise ValueError("dev_names deve ter um nome por equipamento.")

    # Encontrar o índice do equipamento
    if isinstance(device, str):
        exact_matches = [
            j for j, name in enumerate(dev_names)
            if name.lower() == device.lower()
        ]

        if exact_matches:
            j = exact_matches[0]
        else:
            partial_matches = [
                j for j, name in enumerate(dev_names)
                if device.lower() in name.lower()
            ]

            if len(partial_matches) == 0:
                raise ValueError(f"Device '{device}' não encontrado.")

            if len(partial_matches) > 1:
                matching_names = [dev_names[k] for k in partial_matches]
                raise ValueError(
                    f"Vários devices correspondem a '{device}': "
                    f"{matching_names}"
                )

            j = partial_matches[0]

    elif isinstance(device, (int, np.integer)):
        j = int(device)

        if not 0 <= j < len(dev_names):
            raise IndexError(
                f"Índice deve estar entre 0 e {len(dev_names) - 1}."
            )

    else:
        raise TypeError(
            "device deve ser o nome ou índice inteiro do equipamento."
        )

    if Type[j] != 1:
        raise ValueError(
            f"{dev_names[j]} não é power-flexible (Type={Type[j]})."
        )

    T = W_matrix.shape[0]
    time_hours = np.arange(T) / 60.0

    wp_min = float(W_P_min[j])
    wp_max = float(W_P_max[j])

    fig, ax_weight = plt.subplots(figsize=figsize)

    ax_weight.plot(
        time_hours,
        W_matrix[:, j],
        color="tab:blue",
        linewidth=5,
        marker=".",
        markersize=2,
        label=r"$W^{\mathrm{FC}}_P(t)$",
        zorder=5
    )

    ax_weight.axhline(
        wp_min,
        color="tab:blue",
        linestyle="--",
        linewidth=1,
        alpha=0.60,
        label=r"$W_{\min}$",
        zorder=2
    )

    ax_weight.axhline(
        wp_max,
        color="tab:blue",
        linestyle="--",
        linewidth=1,
        alpha=0.60,
        label=r"$W_{\max}$",
        zorder=2
    )

    weight_margin = max(0.5, 0.10 * abs(wp_max - wp_min))
    weight_ymin = wp_min - weight_margin
    weight_ymax = wp_max + weight_margin

    ax_weight.set_ylim(weight_ymin, weight_ymax)
    ax_weight.set_xlim(0, T / 60.0)
    ax_weight.set_xticks(np.arange(0, 25, 1))
    ax_weight.set_xlabel("Hora do dia")
    ax_weight.set_ylabel(r"Power weight $W^{\mathrm{FC}}_P(t)$")
    ax_weight.grid(axis="both", alpha=0.25, zorder=1)

    # Segundo eixo: ON/OFF e Y
    ax_state = ax_weight.twinx()

    # width=1/60: cada barra ocupa um minuto no eixo em horas
    ax_state.bar(
        time_hours,
        X_matrix[:, j],
        width=1 / 60.0,
        alpha=0.25,
        color="tab:blue",
        label="ON/OFF state",
        zorder=2,
        align="edge"
    )

    # Y fica sempre visível e desce a zero quando o equipamento desliga
    Y_plot = np.where(
        X_matrix[:, j] == 1,
        Y_matrix[:, j],
        0.0
    )

    ax_state.step(
        time_hours,
        Y_plot,
        where="post",
        color="tab:orange",
        linewidth=1.2,
        label=r"$Y$",
        zorder=3
    )

    # Alinhamento: OFF ↔ W_P,min; ON ↔ W_P,max
    weight_range = weight_ymax - weight_ymin
    off_position = (wp_min - weight_ymin) / weight_range
    on_position = (wp_max - weight_ymin) / weight_range

    state_range = 1.0 / (on_position - off_position)
    state_ymin = -off_position * state_range
    state_ymax = state_ymin + state_range

    ax_state.set_ylim(state_ymin, state_ymax)
    ax_state.set_yticks([0, 1])
    ax_state.set_yticklabels(["OFF", "ON"])
    ax_state.set_ylabel(r"ON/OFF state and power mode $Y$")
    ax_state.grid(False)

    ax_weight.set_title(
        f"Power weight, Y and activation — "
        f"{characterization_mode} - {dev_names[j]}"
    )

    ax_weight.title.set_fontsize(18)
    ax_weight.xaxis.label.set_size(16)
    ax_weight.yaxis.label.set_size(16)
    ax_state.yaxis.label.set_size(16)
    ax_weight.tick_params(axis="both", labelsize=14)
    ax_state.tick_params(axis="y", labelsize=14)

    handles_weight, labels_weight = ax_weight.get_legend_handles_labels()
    handles_state, labels_state = ax_state.get_legend_handles_labels()

    ax_weight.legend(
        handles_weight + handles_state,
        labels_weight + labels_state,
        loc="upper left",
        ncol=2,
        fontsize=14
    )

    plt.tight_layout()
    plt.show()
    return fig, ax_weight

In [ ]:
def _compress_ton_toff_segments(T_ON, T_OFF):
    """
    Compress consecutive samples with the same (T_ON, T_OFF) pair.

    Returns
    -------
    ton_values : list
        One T_ON value per constant segment.

    toff_values : list
        One T_OFF value per constant segment.
    """

    T_ON = np.asarray(T_ON)
    T_OFF = np.asarray(T_OFF)

    if len(T_ON) != len(T_OFF):
        raise ValueError("T_ON and T_OFF must have the same length.")

    if len(T_ON) == 0:
        return [], []

    ton_values = [T_ON[0]]
    toff_values = [T_OFF[0]]

    for t in range(1, len(T_ON)):

        same_pair = (
            T_ON[t] == T_ON[t - 1]
            and T_OFF[t] == T_OFF[t - 1]
        )

        if not same_pair:
            ton_values.append(T_ON[t])
            toff_values.append(T_OFF[t])

    return ton_values, toff_values

def plot_ton_toff_boxplots_by_timezone(
    T_ON_device,
    T_OFF_device,
    device_name,
    timezone_limits=None,
    figsize=(14, 6)
):
    """
    Plot T_ON and T_OFF distributions by timezone for one device.

    Consecutive samples with the same (T_ON, T_OFF) pair are treated
    as a single characterized episode/segment.
    """

    if timezone_limits is None:
        timezone_limits = {
            0: (0, 600),
            1: (600, 1170),
            2: (1170, 1350),
            3: (1350, 1440),
        }

    timezone_names = {
        0: "00:00–10:00",
        1: "10:00–19:30",
        2: "19:30–22:30",
        3: "22:30–00:00",
    }

    fig, ax = plt.subplots(figsize=figsize)

    data = []
    positions = []
    labels = []

    group_centers = []

    spacing = 4

    for z, (start, end) in timezone_limits.items():

        ton_zone = T_ON_device[start:end]
        toff_zone = T_OFF_device[start:end]

        ton_values, toff_values = _compress_ton_toff_segments(
            ton_zone,
            toff_zone
        )

        base = z * spacing

        data.extend([
            ton_values,
            toff_values
        ])

        positions.extend([
            base + 1,
            base + 2
        ])

        labels.extend([
            r"$T_{ON}$",
            r"$T_{OFF}$"
        ])

        group_centers.append(base + 1.5)

    ax.boxplot(
        data,
        positions=positions,
        widths=0.7,
        showmeans=True,
        showfliers=True
    )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)

    ax.set_ylabel("Duration [min]")

    ax.set_title(
        f"$T_{{ON}}$ and $T_{{OFF}}$ distributions by operational period — "
        f"{device_name}"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ymin, ymax = ax.get_ylim()

    for z, center in enumerate(group_centers):
        ax.text(
            center,
            ymin - 0.12 * (ymax - ymin),
            timezone_names[z],
            ha="center",
            va="top",
            fontweight="bold"
        )

    for z in range(1, len(timezone_limits)):
        ax.axvline(
            z * spacing - 0.5,
            linestyle="--",
            alpha=0.3
        )

    plt.subplots_adjust(bottom=0.20)
    plt.show()

In [ ]:
def plot_episode(X, episode, device_name=None):
    """
    Plota um episódio OFF-ON e os respetivos T_OFF e T_ON.

    O eixo temporal é apresentado em horas, mas T_OFF e T_ON
    continuam indicados em minutos.
    """

    start = episode.start
    end = episode.end + 1

    T_OFF = episode.T_OFF
    T_ON = episode.T_ON

    on_samples = np.flatnonzero(np.asarray(X[start:end]) == 1)
    transition = start + int(on_samples[0]) if on_samples.size else end

    # Espaço adicional antes e depois do episódio, em minutos
    margin = max(5, int(0.1 * (end - start)))

    plot_start = max(0, start - margin)
    plot_end = min(len(X), end + margin)

    # Converter minutos para horas para desenhar
    time_hours = np.arange(plot_start, plot_end) / 60.0

    start_h = start / 60.0
    transition_h = transition / 60.0
    end_h = end / 60.0

    fig, ax = plt.subplots(figsize=(15, 5))

    # Estado binário
    ax.step(
        time_hours,
        X[plot_start:plot_end],
        where="post",
        linewidth=2.5,
        label=r"$X(t)$"
    )

    # Períodos OFF e ON
    ax.axvspan(
        start_h,
        transition_h,
        color="tab:blue",
        alpha=0.20,
        label="OFF period"
    )

    ax.axvspan(
        transition_h,
        end_h,
        color="tab:orange",
        alpha=0.20,
        label="ON period"
    )

    # Limites do episódio
    ax.axvline(start_h, linestyle="--", linewidth=1.2)
    ax.axvline(transition_h, linestyle="--", linewidth=1.2)
    ax.axvline(end_h, linestyle="--", linewidth=1.2)

    # Seta T_OFF
    ax.annotate(
        "",
        xy=(transition_h, 0.25),
        xytext=(start_h, 0.25),
        arrowprops=dict(
            arrowstyle="<->",
            linewidth=2,
            color="tab:blue"
        )
    )

    ax.text(
        (start_h + transition_h) / 2,
        0.30,
        rf"$T_{{OFF}}={T_OFF}$ min",
        ha="center",
        va="bottom",
        fontsize=13,
        color="tab:blue"
    )

    # Seta T_ON
    ax.annotate(
        "",
        xy=(end_h, 0.75),
        xytext=(transition_h, 0.75),
        arrowprops=dict(
            arrowstyle="<->",
            linewidth=2,
            color="tab:orange"
        )
    )

    ax.text(
        (transition_h + end_h) / 2,
        0.80,
        rf"$T_{{ON}}={T_ON}$ min",
        ha="center",
        va="bottom",
        fontsize=13,
        color="tab:orange"
    )

    title = "OFF–ON appliance episode"

    if device_name is not None:
        title += f" — {device_name}"

    ax.set_title(title, fontsize=15)
    ax.set_xlabel("Hora do dia")
    ax.set_ylabel(r"Activation state $X(t)$")

    ax.set_yticks([0, 1])
    ax.set_yticklabels(["OFF", "ON"])
    ax.set_ylim(-0.15, 1.15)

    # Limites exatos da janela do episódio
    x_start = plot_start / 60.0
    x_end = (plot_end - 1) / 60.0

    ax.set_xlim(x_start, x_end)

    # Ticks apenas nas horas que estão dentro da janela desenhada
    first_hour = int(np.ceil(x_start))
    last_hour = int(np.floor(x_end))

    if first_hour <= last_hour:
        ax.set_xticks(np.arange(first_hour, last_hour + 1))

    ax.grid(axis="x", alpha=0.25)
    ax.legend(loc="upper left")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_inside_boundary_comparison(
    W_original,
    W_transition,
    Type,
    dev_names,
    tol=1e-6,
    figsize=(14, 6)
):
    """
    Compares the percentage of time spent inside the open weight
    interval 1 < WT < 10 for two characterization methods.
    """

    original = compute_inside_boundary_percentage(
        W_original,
        Type,
        dev_names,
        tol
    )

    transition = compute_inside_boundary_percentage(
        W_transition,
        Type,
        dev_names,
        tol
    )

    comparison = original.merge(
        transition,
        on="Device",
        suffixes=("_original", "_transition")
    )

    comparison = comparison.rename(columns={
        "Inside boundary [%]_original": "Operational-Period method",
        "Inside boundary [%]_transition": "Episode-based method"
    })

    devices = comparison["Device"].values
    original_values = comparison["Operational-Period method"].values
    transition_values = comparison["Episode-based method"].values

    x = np.arange(len(devices))
    width = 0.38

    fig, ax = plt.subplots(figsize=figsize)

    bars_original = ax.bar(
        x - width / 2,
        original_values,
        width,
        label="Operational-Period method"
    )

    bars_transition = ax.bar(
        x + width / 2,
        transition_values,
        width,
        label="Episode-based method"
    )

    ax.set_ylabel(r"Time within $1 < W_T < 10$ [%]")
    ax.set_xlabel("Appliance")
    ax.set_title(
        "Comparison of time-weight characterization methods"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        devices,
        rotation=35,
        ha="right"
    )

    ax.set_ylim(0, 105)
    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.legend()

    # Add values above the bars
    ax.bar_label(
        bars_original,
        fmt="%.1f",
        padding=3,
        fontsize=9
    )

    ax.bar_label(
        bars_transition,
        fmt="%.1f",
        padding=3,
        fontsize=9
    )

    plt.tight_layout()
    plt.show()

    return comparison

In [ ]:
def compute_inside_boundary_percentage(
    W_matrix,
    Type,
    dev_names,
    tol=1e-6
):
    """
    Computes the percentage of valid samples for which:

        1 < WT < 10

    for every time-flexible appliance.
    """

    rows = []

    for j in np.where(Type == 2)[0]:

        W = np.asarray(W_matrix[:, j], dtype=float)

        # Remove possible NaN values
        W = W[~np.isnan(W)]

        if len(W) == 0:
            inside_percentage = np.nan
        else:
            at_min = np.isclose(W, 1.0, atol=tol)
            at_max = np.isclose(W, 10.0, atol=tol)

            inside = ~(at_min | at_max)

            inside_percentage = 100 * np.mean(inside)

        rows.append({
            "Device": dev_names[j],
            "Inside boundary [%]": inside_percentage
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_device_weight_and_activation(
    device,
    W_matrix,
    X_matrix,
    dev_names,
    characterization_mode,
    Type=None,
    figsize=(16, 5)
):
    """
    Plota o peso temporal W_T e o estado ON/OFF de um device
    time-flexible, usando horas no eixo temporal.
    """

    W_matrix = np.asarray(W_matrix, dtype=float)
    X_matrix = np.asarray(X_matrix)

    if W_matrix.ndim != 2:
        raise ValueError("W_matrix deve ter shape (T, N_DEV).")

    if X_matrix.shape != W_matrix.shape:
        raise ValueError("X_matrix deve ter o mesmo shape de W_matrix.")

    if len(dev_names) != W_matrix.shape[1]:
        raise ValueError("dev_names deve ter um nome por equipamento.")

    # Encontrar o índice do equipamento
    if isinstance(device, str):
        exact_matches = [
            j for j, name in enumerate(dev_names)
            if name.lower() == device.lower()
        ]

        if exact_matches:
            j = exact_matches[0]
        else:
            partial_matches = [
                j for j, name in enumerate(dev_names)
                if device.lower() in name.lower()
            ]

            if len(partial_matches) == 0:
                raise ValueError(f"Device '{device}' não encontrado.")

            if len(partial_matches) > 1:
                matching_names = [dev_names[k] for k in partial_matches]

                raise ValueError(
                    f"Vários devices correspondem a '{device}': "
                    f"{matching_names}"
                )

            j = partial_matches[0]

    elif isinstance(device, (int, np.integer)):
        j = int(device)

        if not 0 <= j < len(dev_names):
            raise IndexError(
                f"Índice deve estar entre 0 e {len(dev_names) - 1}."
            )

    else:
        raise TypeError(
            "device deve ser o nome ou índice inteiro do equipamento."
        )

    if Type is not None:
        Type = np.asarray(Type)

        if Type[j] != 2:
            raise ValueError(
                f"{dev_names[j]} não é time-flexible (Type={Type[j]})."
            )

    T = W_matrix.shape[0]
    time_hours = np.arange(T) / 60.0

    fig, ax_weight = plt.subplots(figsize=figsize)

    # Peso temporal W_T
    ax_weight.plot(
        time_hours,
        W_matrix[:, j],
        color="tab:blue",
        linewidth=2,
        label=r"$W^{\mathrm{FC}}_T(t)$",
        zorder=4
    )

    ax_weight.axhline(
        1,
        color="tab:blue",
        linestyle="--",
        linewidth=1,
        alpha=0.60,
        label=r"$W_{\min} = 1$",
        zorder=2
    )

    ax_weight.axhline(
        10,
        color="tab:blue",
        linestyle="--",
        linewidth=1,
        alpha=0.60,
        label=r"$W_{\max} = 10$",
        zorder=2
    )

    # nearest can assign a boundary duration shorter than its ON block.
    # Show the raw weight even when it leaves the reference interval.
    finite_weights = W_matrix[np.isfinite(W_matrix[:, j]), j]
    weight_ymin = min(0.5, finite_weights.min() - 0.5) if finite_weights.size else 0.5
    weight_ymax = max(10.5, finite_weights.max() + 0.5) if finite_weights.size else 10.5

    ax_weight.set_ylim(weight_ymin, weight_ymax)
    ax_weight.set_xlim(0, T / 60.0)
    ax_weight.set_xticks(np.arange(0, 25, 1))
    ax_weight.set_xlabel("Hora do dia")
    ax_weight.set_ylabel(r"Time weight $W^{\mathrm{FC}}_T(t)$")
    if weight_ymin == 0.5 and weight_ymax == 10.5:
        ax_weight.set_yticks(np.arange(1, 11))
    ax_weight.grid(axis="both", alpha=0.25, zorder=1)

    # Eixo ON/OFF
    ax_state = ax_weight.twinx()

    # width=1/60 porque cada amostra corresponde a um minuto
    ax_state.bar(
        time_hours,
        X_matrix[:, j],
        width=1 / 60.0,
        align="edge",
        alpha=0.20,
        color="tab:blue",
        label="ON/OFF state",
        zorder=3
    )

    # Alinhar OFF com W_T=1 e ON com W_T=10
    weight_range = weight_ymax - weight_ymin
    off_position = (1.0 - weight_ymin) / weight_range
    on_position = (10.0 - weight_ymin) / weight_range

    state_range = 1.0 / (on_position - off_position)
    state_ymin = -off_position * state_range
    state_ymax = state_ymin + state_range

    ax_state.set_ylim(state_ymin, state_ymax)
    ax_state.set_yticks([0, 1])
    ax_state.set_yticklabels(["OFF", "ON"])
    ax_state.set_ylabel("ON/OFF state")
    ax_state.grid(False)

    ax_weight.set_title(
        f"Time weight and activation — "
        f"{characterization_mode} - {dev_names[j]}"
    )

    ax_weight.title.set_fontsize(18)
    ax_weight.xaxis.label.set_size(16)
    ax_weight.yaxis.label.set_size(16)
    ax_state.yaxis.label.set_size(16)
    ax_weight.tick_params(axis="both", labelsize=14)
    ax_state.tick_params(axis="y", labelsize=14)

    handles_weight, labels_weight = ax_weight.get_legend_handles_labels()
    handles_state, labels_state = ax_state.get_legend_handles_labels()

    ax_weight.legend(
        handles_weight + handles_state,
        labels_weight + labels_state,
        loc="upper left",
        ncol=2,
        fontsize=14
    )

    plt.tight_layout()
    plt.show()
    return fig, ax_weight

In [ ]:
def compute_weight_boundary_metrics(
    W_matrix,
    Type,
    dev_names,
    tol=1e-6
):
    rows = []

    for j in np.where(Type == 2)[0]:

        W_j = W_matrix[:, j]

        valid = ~np.isnan(W_j)
        W_valid = W_j[valid]

        rows.append({
            "Device": dev_names[j],
            "WT = 10 [min]": int(
                np.sum(
                    np.isclose(
                        W_valid,
                        10.0,
                        atol=tol
                    )
                )
            ),
            "WT = 1 [min]": int(
                np.sum(
                    np.isclose(
                        W_valid,
                        1.0,
                        atol=tol
                    )
                )
            ),
            "WT = 10 [%]": (
                100
                * np.mean(
                    np.isclose(
                        W_valid,
                        10.0,
                        atol=tol
                    )
                )
            ),
            "WT = 1 [%]": (
                100
                * np.mean(
                    np.isclose(
                        W_valid,
                        1.0,
                        atol=tol
                    )
                )
            )
        })

    return pd.DataFrame(rows)

# Reference characterization

In [ ]:
def compute_operational_ton_toff_day(j, nome_do_arquivo, diretorio, target_date):
    """
    Operational devices:
    T_ON/T_OFF por timezone, avaliando presença ON dentro da timezone,
    mesmo quando a ativação começou antes da timezone.

    Regras:
      - se não há minutos ON na timezone:
            T_ON  = T_NO_ACTIVITY_FALLBACK
            T_OFF = T_NO_ACTIVITY_FALLBACK

      - se há minutos ON na timezone:
            T_ON = duração total ON dentro da timezone, ou mediana dos blocos ON na timezone

            se houver gaps OFF observados:
                T_OFF = mediana dos gaps OFF
            senão:
                T_OFF = T_OFF_FORCED_ON
    """

    T_ON_day_j = np.full(DAY_MINUTES, T_NO_ACTIVITY_FALLBACK, dtype=int)
    T_OFF_day_j = np.full(DAY_MINUTES, T_NO_ACTIVITY_FALLBACK, dtype=int)

    df_day = load_activation_day_csv(diretorio, nome_do_arquivo, target_date)

    if df_day.empty:
        return T_ON_day_j, T_OFF_day_j

    # construir X_day do device
    X_dev = np.zeros(DAY_MINUTES, dtype=int)

    for _, row in df_day.iterrows():
        start = minute_of_day(row["first_min_on"])
        end = minute_of_day(row["last_min_on"])

        if end >= start:
            X_dev[start:end + 1] = 1
        else:
            X_dev[start:DAY_MINUTES] = 1
            X_dev[0:end + 1] = 1

    zone_limits = {
        0: (0, 600),       # 00:00 - 09:59
        1: (600, 1170),   # 10:00 - 19:29
        2: (1170, 1350),  # 19:30 - 22:29
        3: (1350, 1440),  # 22:30 - 23:59
    }

    zone_ton = {}
    zone_toff = {}

    for z, (start_z, end_z) in zone_limits.items():
        X_zone = X_dev[start_z:end_z]

        on_minutes = int(np.sum(X_zone))

        # sem presença ON na timezone
        if on_minutes == 0:
            zone_ton[z] = T_NO_ACTIVITY_FALLBACK
            zone_toff[z] = T_NO_ACTIVITY_FALLBACK
            continue

        # há presença ON na timezone
        # calcular blocos ON e OFF dentro da timezone
        on_blocks = []
        off_blocks = []

        current_state = X_zone[0]
        count = 1

        for val in X_zone[1:]:
            if val == current_state:
                count += 1
            else:
                if current_state == 1:
                    on_blocks.append(count)
                else:
                    off_blocks.append(count)

                current_state = val
                count = 1

        # fechar último bloco
        if current_state == 1:
            on_blocks.append(count)
        else:
            off_blocks.append(count)

        # T_ON: se há blocos ON, usa mediana dos blocos ON
        if len(on_blocks) > 0:
            ton_z = max(1, int(np.median(on_blocks)))
        else:
            ton_z = T_NO_ACTIVITY_FALLBACK

        # T_OFF:
        # só usar gaps OFF se existirem dentro da zona
        # se está sempre ON ou sem OFF observado, usar forced-on
        valid_off_blocks = [b for b in off_blocks if b > 0]

        if len(valid_off_blocks) > 0:
            toff_z = max(1, int(np.median(valid_off_blocks)))
        else:
            toff_z = T_OFF_FORCED_ON

        zone_ton[z] = ton_z
        zone_toff[z] = toff_z

    for t in range(DAY_MINUTES):
        z = get_time_zone(t)
        T_ON_day_j[t] = zone_ton[z]
        T_OFF_day_j[t] = zone_toff[z]
        
    return T_ON_day_j, T_OFF_day_j

def compute_non_operational_ton_toff_for_window(j, X_window):
    """
    Non-Operational devices:
    Replica a lógica da Ana dentro da janela de 60 min.
    """
    T_ON_window_j = np.zeros(T, dtype=int)
    T_OFF_window_j = np.zeros(T, dtype=int)

    cont_on = 0
    cont_off = 0
    periodo = 0
    first = 0

    for i in range(T):
        if i == 0 and X_window[i] == 0:
            first = 0

        if X_window[i] == 0:
            cont_off += 1

        if X_window[i] == 1:
            cont_on += 1

        if X_window[i] == 1 and i < T - 1 and X_window[i + 1] == 0:
            periodo = i
            for k in range(first, periodo + 1):
                T_OFF_window_j[k] = max(1, cont_off)
                T_ON_window_j[k] = max(1, cont_on)

            cont_on = 1
            cont_off = 1
            first = periodo + 1

        elif X_window[i] == 1 and i == T - 1:
            for k in range(first, T):
                T_OFF_window_j[k] = max(1, cont_off)
                T_ON_window_j[k] = max(1, cont_on)

        else:
            if i == T - 1:
                for k in range(first, T):
                    if X_window[i] == 1:
                        T_ON_window_j[k] = max(1, cont_on)
                        T_OFF_window_j[k] = max(1, T_ON_window_j[k])
                    else:
                        T_OFF_window_j[k] = max(1, cont_off - 1)
                        T_ON_window_j[k] = max(1, T_OFF_window_j[k])

    T_ON_window_j[T_ON_window_j <= 0] = 1
    T_OFF_window_j[T_OFF_window_j <= 0] = 1

    return T_ON_window_j, T_OFF_window_j

def build_X_day_all_devices(day_start_dt, activation_files, diretorio):
    X_day = np.zeros((DAY_MINUTES, N_DEV), dtype=int)
    target_date = day_start_dt.date()

    for j in range(N_DEV):
        build_X_day_for_device(
            j=j,
            nome_do_arquivo=activation_files[j],
            diretorio=diretorio,
            target_date=target_date,
            X_day=X_day
        )

    return X_day

def build_power_flexible_schedule(T, intervals_on):
    x = np.zeros(T, dtype=int)

    for start, end in intervals_on:
        start = max(0, int(start))
        end = min(T, int(end))

        if end > start:
            x[start:end] = 1
        else:  # intervalo que atravessa a meia-noite
            x[start:T] = 1
            x[0:end] = 1

    return x


def get_power_flexible_ton_toff(T, start_minute, end_minute):
    if end_minute > start_minute:
        on_duration = end_minute - start_minute
    else:
        on_duration = (T - start_minute) + end_minute

    off_duration = T - on_duration

    T_ON_j = np.full(T, max(on_duration, 1), dtype=int)
    T_OFF_j = np.full(T, max(off_duration, 1), dtype=int)

    return T_ON_j, T_OFF_j

def build_external_profile_from_activation_file(
    diretorio,
    nome_do_arquivo,
    target_date,
    T=1440
):

    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)

    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    df = pd.read_excel(caminho_arquivo).iloc[:, 0:3].copy()

    df.columns = df.columns.str.strip()

    df["First_min_on"] = pd.to_datetime(
        df["First_min_on"],
        utc=True,
        errors="coerce"
    )

    df["Last_min_on"] = pd.to_datetime(
        df["Last_min_on"],
        utc=True,
        errors="coerce"
    )

    df["First_min_on"] = df["First_min_on"].dt.tz_convert(None)
    df["Last_min_on"] = df["Last_min_on"].dt.tz_convert(None)

    # IMPORTANTÍSSIMO
    target_date = pd.to_datetime(target_date).date()

    df = df[
        df["First_min_on"].dt.date == target_date
    ].copy()

    x = np.zeros(T, dtype=int)

    for _, row in df.iterrows():

        start = minute_of_day(row["First_min_on"])
        end = minute_of_day(row["Last_min_on"])

        if end >= start:
            x[start:end + 1] = 1
        else:
            x[start:T] = 1
            x[0:end + 1] = 1

    return x

In [ ]:
T_ON_FC_matrix = np.ones((T, N_DEV), dtype=int)
T_OFF_FC_matrix = np.ones((T, N_DEV), dtype=int)

# 1) Adicionar devices power-flexible simulados
for j in POWER_FLEXIBLE_DEVICES:

    # só os que têm horário definido manualmente
    if j not in POWER_FLEXIBLE_SCHEDULES:
        continue

    sched = POWER_FLEXIBLE_SCHEDULES[j]

    X_onoff_FC_matrix[:, j] = build_power_flexible_schedule(
        T=T,
        intervals_on=[(sched["start"], sched["end"])]
    )
# 2) Adicionar perfis externos
for j in EXTERNAL_PROFILE_DEVICES:

    X_onoff_FC_matrix[:, j] = build_external_profile_from_activation_file(
        diretorio=diretorio,
        nome_do_arquivo=external_activation_files[j],
        target_date=EXTERNAL_PROFILE_DATES[j],
        T=T
    )
    
# =========================
# BUILD Y_FC_MATRIX
# =========================

Y_FC_matrix = np.zeros((T, N_DEV), dtype=float)

for j in range(N_DEV):

    if Type[j] == 1:  # power-flexible
        Y_FC_matrix[:, j] = X_onoff_FC_matrix[:, j] * Y_max[j]

    else:
        # Para devices não power-flexible, Y não é relevante.
        # Mas pode ficar igual a X para manter consistência.
        Y_FC_matrix[:, j] = X_onoff_FC_matrix[:, j]
        

# 5) Construir T_ON / T_OFF
for j in range(N_DEV):

    if j in POWER_FLEXIBLE_DEVICES:

        if j in POWER_FLEXIBLE_SCHEDULES:

            X_j = X_onoff_FC_matrix[:, j]

            on_minutes = int(np.sum(X_j))
            off_minutes = int(T - on_minutes)

            T_ON_j = np.full(T, max(on_minutes, 1), dtype=int)
            T_OFF_j = np.full(T, max(off_minutes, 1), dtype=int)

        elif j in EXTERNAL_PROFILE_DEVICES:

            on_minutes = int(np.sum(X_onoff_FC_matrix[:, j]))
            off_minutes = T - on_minutes

            T_ON_j = np.full(T, max(on_minutes, 1), dtype=int)
            T_OFF_j = np.full(T, max(off_minutes, 1), dtype=int)

        else:
            T_ON_j = np.ones(T, dtype=int)
            T_OFF_j = np.ones(T, dtype=int)

    elif j in NON_OPERATIONAL_DEVICES:
        T_ON_j, T_OFF_j = compute_non_operational_ton_toff_for_window(
            j,
            X_onoff_FC_matrix[:, j]
        )

    elif j in OPERATIONAL_DEVICES:
        T_ON_j, T_OFF_j = compute_operational_ton_toff_day(
            j=j,
            nome_do_arquivo=activation_files[j],
            diretorio=diretorio,
            target_date=target_date
        )

    else:
        T_ON_j = np.ones(T, dtype=int)
        T_OFF_j = np.ones(T, dtype=int)

    T_ON_FC_matrix[:, j] = T_ON_j
    T_OFF_FC_matrix[:, j] = T_OFF_j

In [ ]:
def compute_time_weight(X, T_ON, T_OFF):
    """
    Computes W(t) according to the paper.

    Parameters
    ----------
    X : ndarray (1440,)
        Binary appliance state.

    T_ON : ndarray (1440,)
    T_OFF : ndarray (1440,)

    Returns
    -------
    W : ndarray (1440,)
    """

    T = len(X)

    W = np.zeros(T)

    # ---------- Initialization ----------

    if X[0] == 0:
        W[0] = 1 + 9 / T_OFF[0]
    else:
        W[0] = 10 - 9 / T_ON[0]

    # ---------- Evolution ----------

    for t in range(1, T):

        if X[t] == 0:

            W[t] = W[t-1] + 9 / T_OFF[t]

        else:

            W[t] = W[t-1] - 9 / T_ON[t]

        W[t] = np.clip(W[t], 1, 10)

    return W

def compute_power_weight(Y, Y_min, Y_max, W_P_min, W_P_max, m=1e-6):
    Y_safe = np.clip(Y, Y_min, Y_max)

    WP = W_P_min + ((W_P_max - W_P_min) * (Y_safe - Y_min)) / (
        (Y_max - Y_min) + m
    )

    return np.clip(WP, min(W_P_min, W_P_max), max(W_P_min, W_P_max))

WT_FC_matrix = np.zeros((T, N_DEV))
WP_FC_matrix = np.zeros((T, N_DEV))

for j in range(N_DEV):

    if Type[j] == 2:  # time-flexible
        WT_FC_matrix[:, j] = compute_time_weight(
            X_onoff_FC_matrix[:, j],
            T_ON_FC_matrix[:, j],
            T_OFF_FC_matrix[:, j]
        )

    elif Type[j] == 1:  # power-flexible
        WP_j = compute_power_weight(
            Y_FC_matrix[:, j],
            Y_min[j],
            Y_max[j],
            Weight_P_min[j],
            Weight_P_max[j]
        )
        WP_FC_matrix[:, j] = np.where(
            X_onoff_FC_matrix[:, j] == 1,
            WP_j,
            np.nan
        )

In [ ]:
metrics_paper = compute_weight_boundary_metrics(
    WT_FC_matrix,
    Type,
    dev_names
)
display(metrics_paper)

# Episode-based characterization

- **Non-Operational** (`NON_OPERATIONAL_DEVICES`): frigoríficos e garde manger, caracterizados no dia inteiro.
- **Operational** (`OPERATIONAL_DEVICES`): dual fryer e salamandra, caracterizados entre a primeira e a última amostra ON, ambas incluídas. Os períodos OFF internos continuam a pertencer à janela.

Dentro da janela aplicam-se as mesmas políticas `observed`, `nearest` e `one`. Se toda a janela contiver apenas um estado, a duração desse estado é a duração da janela e a duração em falta é `MISSING_STATE_DURATION = 1e-6` minutos, independentemente da política. As durações são guardadas como `float`.

Nos equipamentos operational, `T_ON`, `T_OFF` e o peso ficam a `NaN` fora da janela. Sem ativações, não existe janela e os três vetores ficam integralmente a `NaN`. O peso é inicializado na primeira amostra da janela e evolui pela mesma equação, sem clipping. A secção anterior «Caracterização Paper» mantém-se como comparação com o método anterior.

Com `nearest`, o primeiro bloco ON pode herdar uma duração diferente da sua duração observada, produzindo pesos fora de `[1, 10]`. O cálculo mantém esses valores e os gráficos mostram a sua amplitude real.


In [ ]:
@dataclass
class Episode_T:
    """
    OFF-ON appliance episode.

    For complete episodes, both the OFF and ON blocks are observed.

    With nearest, only complete OFF -> ON episodes are returned,
    except for single-state operational windows with an epsilon duration.
    The observed/one policies also return completed boundary segments.

    start
        First observed index belonging to the episode.

    end
        Last observed index belonging to the episode.

    T_OFF
        Observed or estimated OFF duration.

    T_ON
        Observed or estimated ON duration.

    timezone
        Timezone associated with the episode.
    """

    start: int
    end: int

    T_OFF: float
    T_ON: float

    timezone: int


def get_operation_window(X):
    """Return first-to-last ON sample as a slice (stop is exclusive).

    No ON samples means an empty window. Internal OFF blocks are retained.
    """
    X = np.asarray(X)
    if X.ndim != 1 or not np.all(np.isin(X, [0, 1])):
        raise ValueError("X must be a one-dimensional binary profile.")
    active = np.flatnonzero(X == 1)
    if active.size == 0:
        return slice(0, 0)
    return slice(int(active[0]), int(active[-1]) + 1)


def compute_episodes_T(
    X,
    incomplete_policy="observed",
    minimum_duration=1, 
    index_offset=0,
    timezone=0,
    operational=False,
    missing_duration=1e-6
):
    """
    Extract OFF-ON episodes and complete boundary episodes according
    to an incomplete-episode policy.

    Complete episode
    ----------------
    A complete episode consists of:

        OFF block -> ON block

    Its durations are assigned directly:

        T_OFF = observed OFF-block duration
        T_ON  = observed ON-block duration

    Incomplete episodes
    -------------------
    Initial ON block:
        The preceding OFF block is not observed.

    Final OFF block:
        The following ON block is not observed.

    Policies
    --------
    observed:
        The missing duration is set equal to the duration of the
        observed block.

    nearest:
        Only complete OFF -> ON episodes are returned. Boundary samples
        inherit BOTH durations from the nearest complete episode in
        compute_ton_toff. Boundary stretches are not counted as episodes.

    one:
        The missing duration is set to minimum_duration.

    Parameters
    ----------
    X : array-like, shape (T,)
        Binary appliance-state vector.

    index_offset : int, default=0
        Offset used to convert local indices into global indices.

    incomplete_policy : {"observed", "nearest", "one"},
        default="observed"
        Policy used to complete boundary episodes.

    minimum_duration : int, default=1
        Minimum accepted duration for ordinary episodes.

    operational : bool, default=False
        Restrict episodes to the first-to-last ON window. If this window
        has only one state, return one episode with missing_duration for
        the absent state, including when the policy is nearest.

    missing_duration : float, default=1e-6
        Positive numerical substitute for a state absent from the entire
        operational window. Does not replace durations in ordinary cycles.

    Returns
    -------
    episodes : list of Episode_T
        Complete episodes for nearest; complete episodes and completed
        boundary segments for observed/one, ordered in time.
    """

    X = np.asarray(X, dtype=int)

    if X.ndim != 1:
        raise ValueError(
            "X must be one-dimensional."
        )

    if not np.all(np.isin(X, [0, 1])):
        raise ValueError(
            "X must contain only 0 and 1."
        )

    valid_policies = {
        "observed",
        "nearest",
        "one",
    }

    if incomplete_policy not in valid_policies:
        raise ValueError(
            "incomplete_policy must be "
            "'observed', 'nearest' or 'one'."
        )

    if minimum_duration < 1:
        raise ValueError(
            "minimum_duration must be at least 1."
        )

    if not np.isfinite(missing_duration) or missing_duration <= 0:
        raise ValueError("missing_duration must be finite and positive.")

    if operational:
        window = get_operation_window(X)
        X = X[window]
        index_offset += window.start

    T = len(X)

    if T == 0:
        return []

    # An all-ON operational window has no complete OFF -> ON episode.
    # Preserve its observed duration; the absent state receives epsilon.
    if operational and np.all(X == X[0]):
        return [Episode_T(
            start=index_offset,
            end=index_offset + T - 1,
            T_OFF=float(T) if X[0] == 0 else float(missing_duration),
            T_ON=float(T) if X[0] == 1 else float(missing_duration),
            timezone=timezone,
        )]

    # --------------------------------------------------
    # 1. Extract consecutive ON/OFF blocks
    # --------------------------------------------------
    runs = []

    run_start = 0
    current_state = X[0]

    for t in range(1, T):

        if X[t] != current_state:

            run_end = t - 1

            runs.append({
                "state": int(current_state),
                "start": int(run_start),
                "end": int(run_end),
                "duration": int(
                    run_end - run_start + 1
                ),
            })

            run_start = t
            current_state = X[t]

    # Close final block.
    run_end = T - 1

    runs.append({
        "state": int(current_state),
        "start": int(run_start),
        "end": int(run_end),
        "duration": int(
            run_end - run_start + 1
        ),
    })

    # --------------------------------------------------
    # 2. Identify complete OFF -> ON episodes
    # --------------------------------------------------
    complete_episodes = []

    for run_index in range(len(runs) - 1):

        off_run = runs[run_index]
        on_run = runs[run_index + 1]

        if (
            off_run["state"] == 0
            and on_run["state"] == 1
        ):
            complete_episodes.append(
                Episode_T(
                    start=(
                        index_offset
                        + off_run["start"]
                    ),
                    end=(
                        index_offset
                        + on_run["end"]
                    ),
                    T_OFF=max(
                        minimum_duration,
                        off_run["duration"]
                    ),
                    T_ON=max(
                        minimum_duration,
                        on_run["duration"]
                    ),
                    timezone=timezone,
                )
            )

    if incomplete_policy == "nearest":
        return complete_episodes

    episodes = complete_episodes.copy()

    # The observed/one policies retain their existing boundary treatment.
    def complete_missing_duration(
        observed_duration,
        missing_state,
        local_start,
        local_end
    ):
        if incomplete_policy == "observed":
            return max(minimum_duration, observed_duration)
        return minimum_duration

    # --------------------------------------------------
    # 3. Initial ON block
    # --------------------------------------------------
    first_run = runs[0]

    if first_run["state"] == 1:

        observed_T_ON = max(
            minimum_duration,
            first_run["duration"]
        )

        estimated_T_OFF = (
            complete_missing_duration(
                observed_duration=observed_T_ON,
                missing_state=0,
                local_start=first_run["start"],
                local_end=first_run["end"]
            )
        )

        episodes.append(
            Episode_T(
                start=(
                    index_offset
                    + first_run["start"]
                ),
                end=(
                    index_offset
                    + first_run["end"]
                ),
                T_OFF=int(estimated_T_OFF),
                T_ON=int(observed_T_ON),
                timezone=timezone,
            )
        )

    # --------------------------------------------------
    # 4. Final OFF block
    # --------------------------------------------------
    last_run = runs[-1]

    if last_run["state"] == 0:

        observed_T_OFF = max(
            minimum_duration,
            last_run["duration"]
        )

        estimated_T_ON = (
            complete_missing_duration(
                observed_duration=observed_T_OFF,
                missing_state=1,
                local_start=last_run["start"],
                local_end=last_run["end"]
            )
        )

        episodes.append(
            Episode_T(
                start=(
                    index_offset
                    + last_run["start"]
                ),
                end=(
                    index_offset
                    + last_run["end"]
                ),
                T_OFF=int(observed_T_OFF),
                T_ON=int(estimated_T_ON),
                timezone=timezone,
            )
        )

    # Keep chronological order.
    episodes.sort(
        key=lambda episode: episode.start
    )

    return episodes

In [ ]:
def compute_ton_toff(
    X,
    index_offset=0,
    incomplete_policy="observed",
    minimum_duration=1,
    operational=False,
    missing_duration=1e-6
):
    """
    Compute T_ON and T_OFF on the daily or operational horizon.

    Operational devices have float durations and NaN outside the first-to-
    last ON window. With no activations, both vectors are entirely NaN.
    A single-state window uses missing_duration for the absent state.

    With nearest, initial ON and final OFF stretches inherit BOTH
    durations from the closest complete OFF -> ON episode. The observed
    ON/OFF profile is preserved. A nonempty profile with no complete
    episode has no nearest reference and raises ValueError for non-operational
    devices. The single-state operational fallback applies to all policies.

    Parameters
    ----------
    X : ndarray
        Binary ON/OFF trajectory.

    Returns
    -------
    T_ON : ndarray
        ON duration assigned to every sample.

    T_OFF : ndarray
        OFF duration assigned to every sample.
    """

    episodes = compute_episodes_T(
        X=X,
        index_offset=index_offset,
        incomplete_policy=incomplete_policy,
        minimum_duration=minimum_duration,
        operational=operational,
        missing_duration=missing_duration,
    )

    T = len(X)

    window = get_operation_window(X) if operational else slice(0, T)
    fill_value = np.nan if operational else 0.0
    T_ON = np.full(T, fill_value, dtype=float)
    T_OFF = np.full(T, fill_value, dtype=float)

    for episode in episodes:

        start = episode.start - index_offset
        end = episode.end - index_offset + 1

        T_ON[start:end] = episode.T_ON
        T_OFF[start:end] = episode.T_OFF

    if incomplete_policy == "nearest" and window.stop > window.start:
        if not episodes:
            raise ValueError(
                "Cannot assign nearest T_ON/T_OFF: the profile contains "
                "no complete OFF -> ON episode."
            )

        first = episodes[0]
        initial_end = first.start - index_offset
        T_ON[window.start:initial_end] = first.T_ON
        T_OFF[window.start:initial_end] = first.T_OFF

        last = episodes[-1]
        final_start = last.end - index_offset + 1
        T_ON[final_start:window.stop] = last.T_ON
        T_OFF[final_start:window.stop] = last.T_OFF

    return T_ON, T_OFF

In [ ]:
def compute_time_weight_2(X, T_ON, T_OFF):
    """
    Computes W(t) according to the paper.

    Parameters
    ----------
    X : ndarray (1440,)
        Binary appliance state.

    T_ON : ndarray (1440,)
    T_OFF : ndarray (1440,)

    Returns
    -------
    W : ndarray (1440,)
    """

    T = len(X)

    W = np.zeros(T)

    # ---------- Initialization ----------

    if X[0] == 0:
        W[0] = 1 + 9 / T_OFF[0]
    else:
        W[0] = 10 - 9 / T_ON[0]

    # ---------- Evolution ----------

    for t in range(1, T):
        if X[t] == 0:
            W[t] = W[t-1] + 9 / T_OFF[t]
        else:
            W[t] = W[t-1] - 9 / T_ON[t]

    return W

In [ ]:
policies = ["observed", "nearest", "one"]

W_methods = {}
T_ON_methods = {}
T_OFF_methods = {}
episodes_methods = {}

# Slices use original day indices; the stop is exclusive.
characterization_windows = {
    j: (get_operation_window(X_onoff_FC_matrix[:, j])
        if j in OPERATIONAL_DEVICES else slice(0, T))
    for j in range(N_DEV) if Type[j] == 2
}

for policy in policies:

    T_ON = np.zeros((T, N_DEV), dtype=float)
    T_OFF = np.zeros((T, N_DEV), dtype=float)
    W = np.zeros((T, N_DEV), dtype=float)
    episodes_by_device = {}

    for j, window in characterization_windows.items():
        X_j = X_onoff_FC_matrix[:, j]
        operational = j in OPERATIONAL_DEVICES

        T_ON[:, j], T_OFF[:, j] = compute_ton_toff(
            X=X_j,
            incomplete_policy=policy,
            operational=operational,
            missing_duration=MISSING_STATE_DURATION,
        )
        episodes_by_device[j] = compute_episodes_T(
            X=X_j,
            incomplete_policy=policy,
            operational=operational,
            missing_duration=MISSING_STATE_DURATION,
        )

        # Initialize and evolve on the selected window, never across NaN.
        W[:, j] = np.nan
        if window.stop > window.start:
            W[window, j] = compute_time_weight_2(
                X_j[window],
                T_ON[window, j],
                T_OFF[window, j],
            )

    W_methods[policy] = W
    T_ON_methods[policy] = T_ON
    T_OFF_methods[policy] = T_OFF
    episodes_methods[policy] = episodes_by_device

W_observed = W_methods["observed"]
W_nearest  = W_methods["nearest"]
W_one      = W_methods["one"]


In [ ]:
metrics_observed = compute_weight_boundary_metrics(
    W_observed,
    Type,
    dev_names
)
metrics_nearest = compute_weight_boundary_metrics(
    W_nearest,
    Type,
    dev_names
)
metrics_one = compute_weight_boundary_metrics(
    W_one,
    Type,
    dev_names
)
display(metrics_observed)
display(metrics_nearest)
display(metrics_one)

# Characterization plots

## Episode example


In [ ]:
j = 0

X = X_onoff_FC_matrix[:, j]

episodes = episodes_methods["nearest"][j]

if episodes:
    episode = episodes[0]
    plot_episode(
        X=X,
        episode=episode,
        device_name=dev_names[j]
    )
else:
    print(f"{dev_names[j]}: sem episódios na região de operação.")

In [ ]:
plot_device_weight_and_activation(
        device="Drinks Refrigerator",
        W_matrix=W_observed,
        characterization_mode="Episode characterization",
        X_matrix=X_onoff_FC_matrix,
        dev_names=dev_names,
        Type=Type
    )

In [ ]:
for device in ("Dual Fryer",):
    plot_device_weight_and_activation(
        device=device,
        W_matrix=W_observed,
        characterization_mode="Episode characterization",
        X_matrix=X_onoff_FC_matrix,
        dev_names=dev_names,
        Type=Type
    )


In [ ]:
plot_power_flexible_device_weight_and_activation(
    device="Light1",
    W_matrix=WP_FC_matrix,
    X_matrix=X_onoff_FC_matrix,
    Y_matrix=Y_FC_matrix,
    dev_names=dev_names,
    characterization_mode="Fixed characterization",
    Type=Type,
    W_P_min=Weight_P_min,
    W_P_max=Weight_P_max
)

# Forecast weights at minute and block resolutions

The minute-level recurrence represents the forecast operating condition used at the real-time resolution, $W^{\mathrm{FC}}_T(t)$. The block recurrence uses state-specific mean rates (equivalent to harmonic-mean durations) and forecast ON/OFF counts to obtain $W^{\mathrm{FC}}_T(k)$ at each block endpoint. These are baseline weights, not optimized real-time decisions. The logic below is self-contained and uses the episode characterization already computed in this notebook.

When one state is absent from a block, its rate uses the available episode durations, matching the day-ahead implementation. Its forecast count is zero. No weight clipping is applied.


In [ ]:
# Baseline minute-level characterization already evaluated above (no optimization).
BLOCK_SIZE = 10
N_BLOCKS = T // BLOCK_SIZE
W_MIN, W_MAX = 1.0, 10.0
assert T % BLOCK_SIZE == 0
T_ON_FC_matrix = T_ON_methods["observed"]
T_OFF_FC_matrix = T_OFF_methods["observed"]
WT_FC_matrix = W_observed

time_flex_available_matrix = np.zeros((T, N_DEV), dtype=bool)
for j, window in characterization_windows.items():
    time_flex_available_matrix[window, j] = True


In [ ]:
def compute_WT_from_alpha_beta(
    X_matrix,
    alpha_matrix,
    beta_matrix,
    available_matrix,
    Type,
    block_size
):

    WT_block = np.full(
        (N_BLOCKS, N_DEV),
        np.nan,
        dtype=float
    )

    for j in range(N_DEV):

        if Type[j] != 2:
            continue

        initialized = False
        W_previous = np.nan

        for k in range(N_BLOCKS):

            start = k * block_size
            end = start + block_size

            available = available_matrix[
                start:end,
                j
            ]

            n_available = int(np.sum(available))

            # Fora da janela operational, WT permanece NaN.
            if n_available == 0:
                continue

            X_block = X_matrix[start:end, j]
            X_available = X_block[available]

            if not initialized:

                first_state = int(X_available[0])

                W_previous = (
                    W_MIN
                    if first_state == 0
                    else W_MAX
                )

                initialized = True

            n_on = int(np.sum(X_available))
            n_off = n_available - n_on

            W_current = (
                W_previous
                + n_off * alpha_matrix[k, j]
                - n_on * beta_matrix[k, j]
            )

            WT_block[k, j] = W_current
            W_previous = W_current

    return WT_block

In [ ]:
TF_available_minutes = np.zeros(
    (N_BLOCKS, N_DEV),
    dtype=int
)

TF_block_available = np.zeros(
    (N_BLOCKS, N_DEV),
    dtype=bool
)

alpha_block = np.full(
    (N_BLOCKS, N_DEV),
    np.nan,
    dtype=float
)

beta_block = np.full(
    (N_BLOCKS, N_DEV),
    np.nan,
    dtype=float
)

for j in range(N_DEV):

    if Type[j] != 2:
        continue

    for k in range(N_BLOCKS):

        start = k * BLOCK_SIZE
        end = start + BLOCK_SIZE

        X_block = X_onoff_FC_matrix[start:end, j]
        T_ON_block = T_ON_FC_matrix[start:end, j]
        T_OFF_block = T_OFF_FC_matrix[start:end, j]

        available = time_flex_available_matrix[
            start:end,
            j
        ]

        n_available = int(np.sum(available))

        TF_available_minutes[k, j] = n_available
        TF_block_available[k, j] = n_available > 0

        # Fora da janela operational:
        # alpha e beta permanecem NaN.
        if n_available == 0:
            continue

        valid_ton = (
            available
            & np.isfinite(T_ON_block)
            & (T_ON_block > 0)
        )

        valid_toff = (
            available
            & np.isfinite(T_OFF_block)
            & (T_OFF_block > 0)
        )

        if not np.any(valid_ton):
            raise RuntimeError(
                f"No valid T_ON for device {j}, block {k}."
            )

        if not np.any(valid_toff):
            raise RuntimeError(
                f"No valid T_OFF for device {j}, block {k}."
            )

        # Usar preferencialmente os minutos em que cada taxa
        # intervém na trajetória forecast.
        mask_on = valid_ton & (X_block == 1)
        mask_off = valid_toff & (X_block == 0)

        # Se o bloco só observar um estado, T_ON/T_OFF continua
        # definido pela caracterização do episódio.
        beta_source = (
            mask_on
            if np.any(mask_on)
            else valid_ton
        )

        alpha_source = (
            mask_off
            if np.any(mask_off)
            else valid_toff
        )

        beta_block[k, j] = np.mean(
            9.0 / T_ON_block[beta_source]
        )

        alpha_block[k, j] = np.mean(
            9.0 / T_OFF_block[alpha_source]
        )

In [ ]:
W_MIN = 1.0
W_MAX = 10.0
DELTA_W = W_MAX - W_MIN

WT_initial = np.full(
    N_DEV,
    np.nan,
    dtype=float
)

for j in range(N_DEV):

    if Type[j] != 2:
        continue

    available_indices = np.flatnonzero(
        time_flex_available_matrix[:, j]
    )

    if len(available_indices) == 0:
        raise RuntimeError(
            f"No available characterization window for device {j}."
        )

    first_available = int(available_indices[0])

    first_state = int(
        X_onoff_FC_matrix[first_available, j]
    )

    WT_initial[j] = (
        W_MIN
        if first_state == 0
        else W_MAX
    )

WT_block_alpha_beta = compute_WT_from_alpha_beta(
    X_matrix=X_onoff_FC_matrix,
    alpha_matrix=alpha_block,
    beta_matrix=beta_block,
    available_matrix=time_flex_available_matrix,
    Type=Type,
    block_size=BLOCK_SIZE,
)

In [ ]:
def plot_device_weight_aggregation(
    device, W_matrix, W_block, X_matrix, dev_names,
    block_size=10, figsize=(16, 5),
):
    """Plot post-minute weights and independently computed end-of-block weights.

    W_block must contain the block recurrence results, not block averages.
    Row t is the state after minute t; it is displayed at (t + 1)/60 h.
    """
    W_matrix = np.asarray(W_matrix, dtype=float)
    W_block = np.asarray(W_block, dtype=float)
    X_matrix = np.asarray(X_matrix)
    if W_matrix.ndim != 2 or X_matrix.shape != W_matrix.shape:
        raise ValueError("Minute weights and activation must have matching (T, N) shapes.")
    T, N = W_matrix.shape
    if len(dev_names) != N:
        raise ValueError("Provide one name per device.")
    if not isinstance(block_size, (int, np.integer)) or block_size <= 0 or T % block_size:
        raise ValueError("block_size must be a positive integer dividing the horizon.")
    if W_block.shape != (T // block_size, N):
        raise ValueError("W_block must have shape (T // block_size, N).")
    if isinstance(device, str):
        matches = [i for i, name in enumerate(dev_names) if name.casefold() == device.casefold()]
        if len(matches) != 1:
            raise ValueError("Provide an unambiguous full device name.")
        j = matches[0]
    elif isinstance(device, (int, np.integer)) and 0 <= device < N:
        j = int(device)
    else:
        raise ValueError("Invalid device index or name.")

    t = (np.arange(T) + 1) / 60
    tk = (np.arange(T // block_size) + 1) * block_size / 60
    values = np.r_[W_matrix[:, j], W_block[:, j]]
    finite = values[np.isfinite(values)]
    if not finite.size:
        raise ValueError("No finite weights to plot.")
    ymin, ymax = min(0.5, finite.min() - 0.5), max(10.5, finite.max() + 0.5)
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(t, W_matrix[:, j], color="tab:blue", linewidth=2,
            label=r"$W^{\mathrm{FC}}_T(t)$", zorder=4)
    ax.plot(tk, W_block[:, j], color="tab:orange", linestyle="--",
            marker="o", markersize=3.8, markerfacecolor="tab:orange", markeredgecolor="tab:orange", linewidth=1.1,
            label=rf"$W^{{\mathrm{{FC}}}}_T(k)$ ($B={block_size}$ min)", zorder=5)
    for value, label in [(1, r"$W_{\min}=1$"), (10, r"$W_{\max}=10$")]:
        ax.axhline(value, color="tab:blue", linestyle="--", linewidth=1,
                   alpha=0.6, label=label, zorder=2)
    ax.set(xlim=(0, T / 60), ylim=(ymin, ymax), xlabel="Hora do dia",
           ylabel=r"Time weight $W^{\mathrm{FC}}_T$")
    ax.set_xticks(np.arange(0, T / 60 + 0.001, 1))
    if ymin == 0.5 and ymax == 10.5:
        ax.set_yticks(np.arange(1, 11))
    ax.grid(axis="both", alpha=0.25, zorder=1)
    state_ax = ax.twinx()
    state_ax.bar(np.arange(T) / 60, X_matrix[:, j], width=1 / 60,
                 align="edge", alpha=0.20, color="tab:blue",
                 label="ON/OFF state", zorder=3)
    state_ax.set_ylim((ymin - 1) / 9, (ymax - 1) / 9)
    state_ax.set_yticks([0, 1], labels=["OFF", "ON"])
    state_ax.set_ylabel("ON/OFF state")
    state_ax.grid(False)
    ax.set_title(f"Time weight and activation — Episode characterization - {dev_names[j]}")
    ax.title.set_fontsize(18)
    ax.xaxis.label.set_size(16)
    ax.yaxis.label.set_size(16)
    state_ax.yaxis.label.set_size(16)
    ax.tick_params(axis="both", labelsize=14)
    state_ax.tick_params(axis="y", labelsize=14)

    handles, labels = ax.get_legend_handles_labels()
    state_handles, state_labels = state_ax.get_legend_handles_labels()
    ax.legend(handles + state_handles, labels + state_labels,
              loc="upper left", ncol=2, fontsize=14)
    fig.tight_layout()
    return fig, ax


In [ ]:
# Compare independently accumulated block weights with minute-level endpoints.
rows = []
for j, window in characterization_windows.items():
    for k in range(N_BLOCKS):
        indices = np.flatnonzero(time_flex_available_matrix[k*BLOCK_SIZE:(k+1)*BLOCK_SIZE, j])
        if indices.size:
            end = k * BLOCK_SIZE + indices[-1]
            rows.append({"Device": dev_names[j], "Block": k,
                         "Minute weight": WT_FC_matrix[end, j],
                         "Block weight": WT_block_alpha_beta[k, j]})
aggregation_validation = pd.DataFrame(rows)
aggregation_validation["Absolute error"] = abs(aggregation_validation["Block weight"] - aggregation_validation["Minute weight"])
np.testing.assert_allclose(aggregation_validation["Block weight"], aggregation_validation["Minute weight"], rtol=0, atol=1e-9)
display(aggregation_validation.groupby("Device")["Absolute error"].max().to_frame())

fig, ax = plot_device_weight_aggregation(
    device="Refrigerator 3",
    W_matrix=WT_FC_matrix,
    W_block=WT_block_alpha_beta,
    X_matrix=X_onoff_FC_matrix,
    dev_names=dev_names,
    block_size=BLOCK_SIZE,
)
plt.show()
